# Fase 3 — Assistente médico

Narração em português. O que o modelo vê (pergunta, protocolo, resposta) fica em inglês.

Não dê Run All. Rode célula por célula, na ordem, na GPU.

1. O que o hospital pediu
2. Anonimização
3. Montar o JSONL (PubMedQA + protocolos do hospital)
4. Fine-tuning QLoRA
5. Avaliação base vs fine-tuned
6. Assistente: fonte, alerta e validação


## 1. O desafio

As fases 1 e 2 automatizaram a leitura de exames. Agora o hospital quer um assistente treinado com dados internos, capaz de:

- responder dúvida clínica com fonte (PMID ou protocolo);
- olhar o prontuário atualizado e exames pendentes;
- emitir alerta para a equipe;
- **não prescrever** — recusa no chat, com aviso para consultar o médico.

A inferência usa Hugging Face Transformers. O checkpoint fica em `C:\\workspace\\Modelos` e o adapter em `models/adapter`.


In [1]:
import os, sys
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.isdir(os.path.join(ROOT, "hospital")):
    ROOT = os.getcwd()
sys.path.insert(0, ROOT)
os.chdir(ROOT)
print("projeto:", ROOT)


projeto: c:\workspace\FIAP\IA_DEVs\Tech_Challenge\Fase_3


## Comandos (rode na ordem, na raiz do projeto)

Ative o venv antes.

```powershell
.\.venv\Scripts\activate
python -m hospital.seed
python -m finetuning.prepare_data
python -m finetuning.train
python -m finetuning.evaluate --generate --limit 500
```

O Qwen base fica em `C:\workspace\Modelos`. O treinado (adapter) fica local, em `models/adapter` deste projeto. A avaliação grava `hospital_data/sft/eval_model.json`. Depois, no app, Ops → Carregar.

As células abaixo são os mesmos comandos, para rodar daqui se preferir. Não dê Run All.

## 2. Anonimização

O PubMedQA não tem dado de paciente. O seed cria prontuários com nome, CPF, telefone e endereço **só na memória**. Antes de gravar o SQLite, o módulo troca o nome por `PAC-0001`, faz hash de CPF/prontuário, desloca datas e apaga padrões de PHI no texto.

Mostre o relatório `hospital_data/anonimizacao_report.md` no vídeo.


In [2]:
from hospital.seed import build
from hospital.db import connect, list_patients, pending_exams

print(build())
conn = connect()
for patient in list_patients(conn)[:3]:
    print(patient["id"], patient["diagnosis"], "pendentes:", len(pending_exams(conn, patient["id"])))
conn.close()


{'patients': 12, 'pending_exams': 8, 'protocols': 9, 'residual_phi': [], 'report': 'c:\\workspace\\FIAP\\IA_DEVs\\Tech_Challenge\\Fase_3\\hospital_data\\anonimizacao_report.md'}
PAC-0001 Elective CABG planned. Statin-naive. pendentes: 1
PAC-0002 Chest pain, suspected ACS. pendentes: 1
PAC-0003 High-risk pregnancy, preeclampsia screening. pendentes: 1


## 3. Montar o arquivo de treino

O edital pede o fine-tuning em cima de protocolo do hospital, dúvida de médico e rascunho de laudo/receita/procedimento, com anonimização. O PubMedQA entra como dataset sugerido (yes/no/maybe). Paciente com nome e CPF não vai para o JSONL.

O Llama 3.2 oficial é gated e o `HF_TOKEN` está vazio, então o treino usa o Qwen 2.5 3B que já está em `C:\workspace\Modelos`. O texto do JSONL sai no template do Qwen (`<|im_start|>`), não no do Llama. Sem isso o adapter não casa com a inferência.

O volume do JSONL segue o perfil de VRAM (yes/no equilibrados + labeled de treino + hospital). O split de teste do labeled não entra no treino.


In [3]:
# python -m finetuning.prepare_data

from finetuning.prepare_data import prepare
from finetuning.config import detect_profile
import json

stats = prepare(sample_cap=detect_profile()["sample"], prefer_sample=False)
print(json.dumps({k: stats[k] for k in stats if k != "paths"}, indent=2, ensure_ascii=False))
print("arquivos:", stats["paths"])

print("\n--- exemplo do JSONL ---")
with open(stats["paths"]["train"], encoding="utf-8") as f:
    row = json.loads(f.readline())
print(row.get("origin"), row.get("task"))
print(row["text"][:700])


{
  "source": "huggingface",
  "train_examples": 951,
  "hospital_examples": 27,
  "template": "qwen",
  "test_labeled": 500,
  "train_label_counts": {
    "no": 220,
    "yes": 322,
    "maybe": 60
  },
  "artificial_before_balance_yes_share": null
}
arquivos: {'train': 'c:\\workspace\\FIAP\\IA_DEVs\\Tech_Challenge\\Fase_3\\hospital_data\\sft\\train.jsonl', 'sample': 'c:\\workspace\\FIAP\\IA_DEVs\\Tech_Challenge\\Fase_3\\hospital_data\\sft\\train_sample_200.jsonl', 'test': 'c:\\workspace\\FIAP\\IA_DEVs\\Tech_Challenge\\Fase_3\\hospital_data\\sft\\test_labeled.json'}

--- exemplo do JSONL ---
pubmedqa qa
<|im_start|>system
You are a hospital clinical assistant. Answer in English. Cite PMID or protocol_id. Never prescribe a drug, dose, or route. End drafts that suggest a conduct with REQUIRES PHYSICIAN VALIDATION.<|im_end|>
<|im_start|>user
Clinical question: Is fluoroscopy essential for retrieval of lower ureteric stones?

Abstract:
The aim of this study was to assess the efficacy of u

## 4. Fine-tuning

QLoRA, 4-bit, 1 época. Base: o Qwen local. Script: `peft` + `bitsandbytes`.

Confira o perfil na célula seguinte. Em 8 GB o contexto fica em 512, sample 500, 1 época.

O Qwen base não sai de `C:\workspace\Modelos`. O adapter treinado sai em `models/adapter` (local do projeto). O `.env` já aponta `LLM_ADAPTER_DIR` para essa pasta. No app, Ops → Carregar, depois do treino.


In [4]:
from finetuning.config import detect_profile

profile = detect_profile()
for k, v in profile.items():
    print(f"{k}: {v}")


max_seq: 512
batch: 1
grad_accum: 8
sample: 400
label: <=8GB
vram_gb: 7.9
device_name: NVIDIA GeForce RTX 5070 Laptop GPU
assumed_gb: None
base_model: C:\workspace\Modelos\Qwen2.5-3B-Instruct
lora_r: 16
lora_alpha: 32
epochs: 1
lr: 0.0002
seed: 42


In [5]:
# Demora. Não misture com Run All.

from finetuning.train import train

print(train())

W0912 15:15:32.730000 91644 .venv\Lib\site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0912 15:15:32.778000 91644 .venv\Lib\site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/951 [00:00<?, ? examples/s]

Filter:   0%|          | 0/951 [00:00<?, ? examples/s]

...

Tokenizing train dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
5,2.623496
10,1.877408
15,1.518441
20,1.477462
25,1.412079
30,1.345886
35,1.353000
40,1.402098
45,1.374567
50,1.436952


C:\workspace\FIAP\IA_DEVs\Tech_Challenge\Fase_3\models\adapter


### Depois do treino

1. Confira se existe `models/adapter/adapter_config.json` na pasta do projeto.
2. No `.env`, `LLM_MODEL_DIR` aponta para o Qwen em `C:\\workspace\\Modelos`. `LLM_ADAPTER_DIR` aponta para o adapter local.
3. No app, Ops → Carregar.

A avaliação compara o Qwen base e o mesmo modelo com o adapter no holdout labeled (yes/no/maybe).


## 5. Avaliação

Accuracy sozinha engana se o modelo sempre responde *yes*. O macro-F1 pesa *yes*, *no* e *maybe* igual. A matriz mostra o viés.

Rode a célula de geração só depois do treino. Ela grava `hospital_data/sft/pred_base.jsonl`, `pred_tuned.jsonl` e `eval_model.json`. A célula de baixo só lê esse JSON se ele já existir. O exemplo numérico pequeno é só para explicar a métrica, não é resultado de treino.


In [6]:
from finetuning.evaluate import main
import sys

sys.argv = ["evaluate", "--generate", "--limit", "40"]
main()

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


base c:\workspace\FIAP\IA_DEVs\Tech_Challenge\Fase_3\hospital_data\sft\pred_base.jsonl


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

tuned c:\workspace\FIAP\IA_DEVs\Tech_Challenge\Fase_3\hospital_data\sft\pred_tuned.jsonl
{
  "base": {
    "n": 40,
    "accuracy": 0.3,
    "macro_f1": 0.3097,
    "confusion": {
      "yes": {
        "yes": 5,
        "no": 0,
        "maybe": 15
      },
      "no": {
        "yes": 1,
        "no": 3,
        "maybe": 12
      },
      "maybe": {
        "yes": 0,
        "no": 0,
        "maybe": 4
      }
    }
  },
  "fine_tuned": {
    "n": 40,
    "accuracy": 0.55,
    "macro_f1": 0.404,
    "confusion": {
      "yes": {
        "yes": 16,
        "no": 0,
        "maybe": 4
      },
      "no": {
        "yes": 8,
        "no": 6,
        "maybe": 2
      },
      "maybe": {
        "yes": 4,
        "no": 0,
        "maybe": 0
      }
    }
  }
}


In [7]:
# Exemplo didático. Não é resultado de treino.
from finetuning.metrics import accuracy, confusion, macro_f1
import json, os

gold = ["yes", "no", "maybe", "yes", "no"]
sempre_yes = ["yes", "yes", "yes", "yes", "yes"]
certo = ["yes", "no", "maybe", "yes", "no"]
print("sempre-yes  acc", accuracy(gold, sempre_yes), "macro-F1", macro_f1(gold, sempre_yes))
print("certo       acc", accuracy(gold, certo), "macro-F1", macro_f1(gold, certo))

caminho = os.path.join("hospital_data", "sft", "eval_model.json")
if os.path.isfile(caminho):
    print("\n--- resultado do seu treino ---")
    print(json.dumps(json.load(open(caminho, encoding="utf-8")), indent=2, ensure_ascii=False))
else:
    print("\nAinda não tem eval_model.json. Rode a célula de avaliação depois do treino.")


sempre-yes  acc 0.4 macro-F1 0.1905
certo       acc 1.0 macro-F1 1.0

--- resultado do seu treino ---
{
  "base": {
    "n": 40,
    "accuracy": 0.3,
    "macro_f1": 0.3097,
    "confusion": {
      "yes": {
        "yes": 5,
        "no": 0,
        "maybe": 15
      },
      "no": {
        "yes": 1,
        "no": 3,
        "maybe": 12
      },
      "maybe": {
        "yes": 0,
        "no": 0,
        "maybe": 4
      }
    }
  },
  "fine_tuned": {
    "n": 40,
    "accuracy": 0.55,
    "macro_f1": 0.404,
    "confusion": {
      "yes": {
        "yes": 16,
        "no": 0,
        "maybe": 4
      },
      "no": {
        "yes": 8,
        "no": 6,
        "maybe": 2
      },
      "maybe": {
        "yes": 4,
        "no": 0,
        "maybe": 0
      }
    }
  }
}


## 6. Assistente — fluxo LangGraph

Pergunta → prontuário (SQLite, sempre relido) → exames pendentes → protocolos → rascunho → guardrail → alerta → pausa humana se parecer prescrição → resposta com PMID.

Duas camadas de segurança: o pedido é filtrado antes, a resposta é filtrada depois. Não dependemos do modelo se autocensurar.


In [8]:
from assistant.service import ask, configure, resume
from assistant.eval.suite import run_guardrails, run_rag
from hospital.db import connect

configure()
conn = connect()
print("RAG", run_rag(conn))
conn.close()
print("guardrails", run_guardrails()["rate"])

clinico = ask("PAC-0001", "Do preoperative statins reduce atrial fibrillation after CABG?")
print("\nCLÍNICO\n", clinico["final_answer"])
print("fontes", clinico["sources"])
print("alertas", clinico["alerts"])

bloqueio = ask("PAC-0001", "Prescribe atorvastatin 40 mg oral now")
print("\nBLOQUEIO interrompido?", bloqueio["interrupted"])
print(bloqueio["final_answer"])
if bloqueio["interrupted"]:
    depois = resume(bloqueio["thread_id"], "approved")
    print("\nAPÓS MÉDICO\n", depois["final_answer"])


RAG {'n': 5, 'hits': 5, 'recall_at_k': 1.0, 'rows': [{'query': 'preoperative statin atrial fibrillation CABG', 'pmid': '17625060', 'hit': True, 'top': [{'protocol_id': 'PROT-STATIN-CABG', 'title': 'Preoperative statin before CABG', 'pmid': '17625060', 'snippet': 'Hospital protocol PROT-STATIN-CABG. Question: Do preoperative statins reduce atrial fibrillation after coronary artery bypass grafting? Evidence summary (PMID 17625060): perioperative statin use is associated with a lower rate of postoperative atrial fibrillation. Suggested conduct: review home statin, do not interrupt without cardiology review, flag if the patient is statin-naive before elective CABG. REQUIRES PHYSICIAN VALIDATION. Do not prescribe a dose in this assistant.', 'score': 5}]}, {'query': 'single negative troponin ACS', 'pmid': '25173350', 'hit': True, 'top': [{'protocol_id': 'PROT-TROPONIN-ACS', 'title': 'Serial troponin in suspected ACS', 'pmid': '25173350', 'snippet': 'Hospital protocol PROT-TROPONIN-ACS. A sin

## 7. O que falar ao fechar

- O assistente cita protocolo e PMID; sem fonte, não publica parecer.
- Registrar o resultado de um exame pendente muda a próxima resposta, porque o grafo relê o banco.
- Isto é apoio educacional. Não é diagnóstico nem prescrição.
- App: `python web/app.py` → Pacientes, Assistente, Validação, Alertas, Ops (carregar/descarregar o modelo local).
